# Introductory Data Science 

<span style="color:purple; font-size: 20px;"> [ World Quant University ] </span> 

## Tidy Data and Scientific Method of CLeaning Data 

### A) Inspecting the Data: The Five-Method Ritual

**The Five-Method Inspection Ritual — run this every time you load a new dataset**
>
> Experienced data scientists have a consistent habit: before touching or modifying
> any new dataset, they run a small set of diagnostic commands to understand its
> shape, types, and completeness. These five checks take about ten seconds in total
> and prevent the vast majority of bugs that come from assumptions about the data.
>
> | Step | Command | Method or Attribute | What it tells you | Significant Extractions & Extrapolations |
> |---|---|---|---|---|
> | 1 | `.head()`  | Method | Visual overview: column names, sample values, index | <ol> <li>Column Names (expected variables)</li><li>Variable naming and type errors at a glace</li><li> Missing values `NaN`</li><li>`lat`/`lon` of cross-sectional data</li> </ol> |
> | 2 | `.shape` | Attribute | Exact dimensions: total rows × total columns | <ol><li> m (rows) x n (columns/no. variables) -> (non-visual representation)</li><li> `.dropna()` reduces m and `.concat()` will increase m by combining clean similar datasets/dataframes/series </li></ol> |
> | 3 | `.info()` | Method | Column types, non-null counts, memory usage — **the most informative single command** |<ol><li> non-null reveals how many rows (observations) are present</li><li> __dtypes__ of variables -> oportunity to identify those that need changing  <ul><li> numbers that are `objects` which should be `float64` or `int64` then we change this with `.astype(float)`</li></ul></li></ol> |
> | 4 | `.dtypes` | Attribute | Data type of each column in isolation — useful for quick type audits | <ol><li>`.info()` makes a long list when you have 50 to 100 columsn so the </li><ol> |
> | 5 | `.isnull().sum()` | Method | Number of missing values per column — reveals where the gaps are | <ol><li>`.isnull()` -> converts every cell in the df into a `bool` = value of **true (1)** or **false (0)** </li><li>`.sum` -> sums the bools col by col. So aggr. of for a bool col will reflect the no. of missing values in that col </li></ol> |
>
> Think of it as the data scientist's equivalent of putting on safety glasses
> before entering the lab. Once you have the results of all five, you can write a
> *specific, informed* cleaning plan rather than guessing.

The code cells below walk through each method. After all five, we synthesize what
their combined output tells us about the Mexico City housing dataset before we
write a single line of cleaning code.

>  **Resist the temptation to skip this step.** Even if you wrote the data
> collection script yourself, always inspect the loaded data. Encoding issues,
> unexpected nulls, and type mismatches appear at the I/O boundary — the gap
> between the file on disk and the object in memory.

In [12]:
import pandas as pd 
import numpy 

In [2]:
path_1 = "./data/mexico-real-estate-1.csv"
path_2 = "./data/mexico-real-estate-2.csv"
path_3 = "./data/mexico-real-estate-3.csv"

print(f'We load data from:\n \n 1 - "{path_1}" \n 2 - "{path_2}" \n 3 - "{path_3}" \n')
print(f'After cleaning our data for each data frame we will have our full clean data set')

We load data from:
 
 1 - "./data/mexico-real-estate-1.csv" 
 2 - "./data/mexico-real-estate-2.csv" 
 3 - "./data/mexico-real-estate-3.csv" 

After cleaning our data for each data frame we will have our full clean data set


In [3]:
(
    pd.read_csv(path_1)
)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150.0,"$67,965.56"
1,house,Nuevo León,25.688436,-100.198807,186.0,"$63,223.78"
2,apartment,Guerrero,16.767704,-99.764383,82.0,"$84,298.37"
3,apartment,Guerrero,16.829782,-99.911012,150.0,"$94,308.80"
4,house,Veracruz de Ignacio de la Llave,NaN,NaN,175.0,"$94,835.67"
...,...,...,...,...,...,...
695,house,Morelos,NaN,NaN,310.0,"$237,089.17"
696,house,Yucatán,21.050653,-89.558841,334.0,"$137,017.34"
697,house,Yucatán,21.343796,-89.262060,130.0,"$110,404.35"
698,apartment,Nuevo León,NaN,NaN,155.0,"$184,446.42"


In [4]:
(
    pd.read_csv(path_1)
    .head()
)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150.0,"$67,965.56"
1,house,Nuevo León,25.688436,-100.198807,186.0,"$63,223.78"
2,apartment,Guerrero,16.767704,-99.764383,82.0,"$84,298.37"
3,apartment,Guerrero,16.829782,-99.911012,150.0,"$94,308.80"
4,house,Veracruz de Ignacio de la Llave,NaN,NaN,175.0,"$94,835.67"


In [5]:
(
    pd.read_csv(path_1)
    .shape 
)

(700, 6)

In [6]:
(
    pd.read_csv(path_1)
    .info()
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  700 non-null    object 
 1   state          700 non-null    object 
 2   lat            583 non-null    float64
 3   lon            583 non-null    float64
 4   area_m2        700 non-null    float64
 5   price_usd      700 non-null    object 
dtypes: float64(3), object(3)
memory usage: 32.9+ KB


In [7]:
(
    pd.read_csv(path_1)
    .dtypes
)

property_type     object
state             object
lat              float64
lon              float64
area_m2          float64
price_usd         object
dtype: object

So now we see that price is an `object` when it is meant to be a `float64` -> likely because we need to strip $ and , characters

In [8]:
(
    pd.read_csv(path_1)
    .isnull()
    .sum()
)

property_type      0
state              0
lat              117
lon              117
area_m2            0
price_usd          0
dtype: int64

So now we see that we need to drop 117 observations from the sample 

### B) Cleaning the Data: Method Chaining 

**What is method chaining and why does it matter?**
>
> In pandas, every transformation method returns a new DataFrame. `.dropna()`
> returns a new DataFrame with certain rows removed. `.assign()` returns a new
> DataFrame with a new or modified column. Because every method returns a DataFrame,
> you can call the next method directly on the result — one after another — without
> storing intermediate results in temporary variables.
>
> This is **method chaining**.

**The "river" mental model:**

Imagine data as water flowing down a river. Each transformation is a section of the
river that filters, shapes, or enriches the water as it flows through:

```python
(
    pd.read_csv(path_1)          # The source: raw water from the spring
    .dropna()                    # First bend: filter out the debris (missing rows)
    .assign(price_usd = ...)     # Second bend: enrich the water (fix the column type)
)
```

The data enters at the top as a raw CSV and exits at the bottom as a clean DataFrame.
Each line adds one transformation. The parentheses `(` and `)` wrapping the entire
chain tell Python to treat this as one statement — without them, Python reads line
by line and raises a `SyntaxError` when it encounters the second method.

**How do you refer to the *current* version of the DataFrame in the middle of a chain?**
>
> Suppose you have cleaned the data with `.dropna()` and now want to create a new
> column based on an existing column. You need to say "take the column as it exists
> *right now* — after the drop, not from the original." A regular function would need
> the cleaned DataFrame passed in as an argument. But inside a chain, you do not
> have a name for that intermediate result. The `lambda` solves this.

**The `lambda x:` pattern — intuition:**

Think of `x` as a placeholder name meaning "whatever the DataFrame looks like right
now, at this exact step in the chain." When `.assign()` sees `lambda x: ...`, it
passes the current state of the DataFrame in as `x`. You can then access any
column with `x["column_name"]` and compute whatever you need.

**Formal definition:**

```python
lambda x: expression
```

A `lambda` is an *anonymous function* — a one-shot function with no name:
- `x` is the **input** — always the DataFrame at the current chain step.
- `expression` is what you compute and **return** — the new column values (as a Series).

In [9]:
(
    pd.read_csv(path_1)
    .dropna()
)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150.0,"$67,965.56"
1,house,Nuevo León,25.688436,-100.198807,186.0,"$63,223.78"
2,apartment,Guerrero,16.767704,-99.764383,82.0,"$84,298.37"
3,apartment,Guerrero,16.829782,-99.911012,150.0,"$94,308.80"
5,house,Yucatán,21.052583,-89.538639,205.0,"$105,191.37"
...,...,...,...,...,...,...
693,house,Puebla,19.051815,-98.277669,198.0,"$115,910.26"
694,apartment,Distrito Federal,19.309208,-99.171906,70.0,"$77,572.89"
696,house,Yucatán,21.050653,-89.558841,334.0,"$137,017.34"
697,house,Yucatán,21.343796,-89.262060,130.0,"$110,404.35"


So now we dropped the observations missing values but we still need to: 
<ol><li>fix <strong> price_usd </strong> by striping the characters $ and ,</li><li>index now has gaps for where the 117 observations were<ul><li>  <strong> df.loc[i] </strong>  returns a  <strong> KeyError  </strong> when i is the index label of a dropped row</li></ul></li></ol>

So lulu =  
```python
`lambda x:` 
```

In [10]:
(
    pd.read_csv(path_1)
    .dropna()
    .assign(
        price_usd = lambda x: 
        x["price_usd"]                          # 1. Select the text column
        .str.replace("$", "", regex=False)      # 2. Remove the dollar sign
        .str.replace(",", "", regex=False)      # 3. Remove the thousands comma
        .astype(float)                          # 4. Convert text to number
    )
)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150.0,67965.56
1,house,Nuevo León,25.688436,-100.198807,186.0,63223.78
2,apartment,Guerrero,16.767704,-99.764383,82.0,84298.37
3,apartment,Guerrero,16.829782,-99.911012,150.0,94308.80
5,house,Yucatán,21.052583,-89.538639,205.0,105191.37
...,...,...,...,...,...,...
693,house,Puebla,19.051815,-98.277669,198.0,115910.26
694,apartment,Distrito Federal,19.309208,-99.171906,70.0,77572.89
696,house,Yucatán,21.050653,-89.558841,334.0,137017.34
697,house,Yucatán,21.343796,-89.262060,130.0,110404.35


**Why `regex=False` is mandatory here**
>
> In regular-expression syntax, `$` means "end of string" — not a literal dollar
> sign. If you omit `regex=False`, pandas treats `"$"` as a regex pattern, finds
> no literal `$` characters to remove (because regex `$` matches positions, not
> characters), and leaves your column dirty — silently.
>
> Similarly, `.` in regex means "any character." Safe habit: whenever you are
> matching a *literal* character with `.str.replace()`, always add `regex=False`.

#### Data Frame 1

In [13]:
# now we just define our 1st cleaned dataset df 1 

df1 = (
    pd.read_csv(path_1)
    .dropna()
    .assign(
        price_usd = lambda x: 
        x["price_usd"]                          # 1. Select the text column
        .str.replace("$", "", regex=False)      # 2. Remove the dollar sign
        .str.replace(",", "", regex=False)      # 3. Remove the thousands comma
        .astype(float)                          # 4. Convert text to number
    )
)

print(f'df1 is type: {type(df1)}')
print(f'df1 has a shape of: {df1.shape}')
df1.head()

df1 is type: <class 'pandas.core.frame.DataFrame'>
df1 has a shape of: (583, 6)


,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150.0,67965.56
1,house,Nuevo León,25.688436,-100.198807,186.0,63223.78
2,apartment,Guerrero,16.767704,-99.764383,82.0,84298.37
3,apartment,Guerrero,16.829782,-99.911012,150.0,94308.80
5,house,Yucatán,21.052583,-89.538639,205.0,105191.37


### Inspecting & Cleaning df2 and df3 

Now we clean the other datasets `mexico-real-estate-2.csv` and `mexico-real-estate-3.csv` — each covering a different set of property listings

In [14]:
(
    pd.read_csv(path_2)
    .head()
)

,property_type,state,lat,lon,area_m2,price_mxn
0,apartment,Nuevo León,25.721081,-100.345581,72.0,1300000.0
1,apartment,Puebla,NaN,NaN,190.0,2500000.0
2,house,Morelos,23.634501,-102.552788,360.0,5300000.0
3,house,Morelos,NaN,NaN,76.0,820000.0
4,house,Puebla,NaN,NaN,200.0,1100000.0


In [15]:
(
    pd.read_csv(path_2)
    .shape
)

(700, 6)

In [16]:
(
    pd.read_csv(path_2)
    .info()
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  700 non-null    object 
 1   state          700 non-null    object 
 2   lat            571 non-null    float64
 3   lon            571 non-null    float64
 4   area_m2        700 non-null    float64
 5   price_mxn      700 non-null    float64
dtypes: float64(4), object(2)
memory usage: 32.9+ KB


In [17]:
(
    pd.read_csv(path_2)
    .dtypes
)

property_type     object
state             object
lat              float64
lon              float64
area_m2          float64
price_mxn        float64
dtype: object

#### DataFrame 2

All we have to change for `path_2` is:  
  
**1.** the Mexican Peso (MXN) -> US Dollar (USD)   
**2.** **Tidy Data** = The same information should not appear in two columns $\therefore$ we create a new `price_usd` column for the dataframe to replace `price_mxn`  
**3.** **`.drop()`** = remove the `price_mxn` column completely from the dataframe because it still exists even after `.assign(price_usd = ...)`  

We do all of this because `pd,.concat()` requres that all 3 DataFrames have the **identical column names in the same order**  
-  A `price_mxn` column in `df2` but not in `df1` or `df3` would cause `pd.concat()` to fill the missing entries with `NaN` — silently producing a corrupted combined dataset.

In [18]:
# cleaning process using method chaining 
df2 = (
    pd.read_csv(path_2)
    .dropna()   
    .assign(
        price_usd = lambda x:
        x["price_mxn"].div(18,33)
    )

    .drop(columns = ["price_mxn"])
)

# print type (confirm we have a DataFrame), shape (confirm the null values have been removed)

print(f'df2 is type: {type(df2)}')
print(f'df2 has a shape of: {df2.shape}')
df2.head()

df2 is type: <class 'pandas.core.frame.DataFrame'>
df2 has a shape of: (571, 6)


,property_type,state,lat,lon,area_m2,price_usd
0,apartment,Nuevo León,25.721081,-100.345581,72.0,72222.222222
2,house,Morelos,23.634501,-102.552788,360.0,294444.444444
6,apartment,Estado de México,19.272040,-99.572013,85.0,69444.444444
7,house,San Luis Potosí,22.138882,-100.996510,158.0,117777.777778
8,apartment,Distrito Federal,19.394558,-99.129707,65.0,42121.666667


#### DataFrame 3

In [19]:
(
    pd.read_csv(path_3)
    .head()
)

,property_type,place_with_parent_names,lat-lon,area_m2,price_usd
0,apartment,|México|Distrito Federal|Gustavo A. Madero|Acu...,"19.52589,-99.151703",71.0,48550.59
1,house,|México|Estado de México|Toluca|Metepec|,"19.2640539,-99.5727534",233.0,168636.73
2,house,|México|Estado de México|Toluca|Toluca de Lerd...,"19.268629,-99.671722",300.0,86932.69
3,house,|México|Morelos|Temixco|Burgos Bugambilias|,NaN,275.0,263432.41
4,apartment,|México|Veracruz de Ignacio de la Llave|Veracruz|,"19.511938,-96.871956",84.0,68508.67


Here we have a different problem, df3 contains the same data on paper but it breaks **tidy data rules**:  
 
1. `lat-lon` -> two values in one cell  
2. `places_with_parent_names` -> variable name embedded in values (state)

so we use `.str.split(separator, expand=True)` - splits the string and expands the result into separate columns 

In [20]:
#inspecting the lat-lon column 

(
    pd.read_csv(path_3)
    ["lat-lon"]
    .head()
)

0       19.52589,-99.151703
1    19.2640539,-99.5727534
2      19.268629,-99.671722
3                       NaN
4      19.511938,-96.871956
Name: lat-lon, dtype: object

In [21]:
(
    pd.read_csv(path_3)
    ["lat-lon"]
    .str.split(",", expand=True)
    .head()
)

,0,1
0,19.52589,-99.151703
1,19.2640539,-99.5727534
2,19.268629,-99.671722
3,NaN,NaN
4,19.511938,-96.871956


**Important:** the values are *still text strings* — they look like numbers but
> are wrapped in quotes by Python. The dtype of both columns is `object`, not `float64`. We must convert them with `.astype(float)` before any > arithmetic is possible. We also want to take lat and lon for our DataFrame as individual columns to match with the other 2 data sets, to do that we access the 0th column indice made by `expand=True` which is equivalent to `df[0]` and the same for the equivalent of `df[1]` using `lambda x:`

In [22]:
(
    pd.read_csv(path_3)
        
    .assign(
    lat = lambda x:
    x["lat-lon"]
    .str.split(",", expand=True)[0]
    .astype(float),

    lon = lambda x: 
    x["lat-lon"]
    .str.split(",", expand=True)[1]
    .astype(float), 
    )

    .drop(columns=["lat-lon"])
)

,property_type,place_with_parent_names,area_m2,price_usd,lat,lon
0,apartment,|México|Distrito Federal|Gustavo A. Madero|Acu...,71.0,48550.59,19.525890,-99.151703
1,house,|México|Estado de México|Toluca|Metepec|,233.0,168636.73,19.264054,-99.572753
2,house,|México|Estado de México|Toluca|Toluca de Lerd...,300.0,86932.69,19.268629,-99.671722
3,house,|México|Morelos|Temixco|Burgos Bugambilias|,275.0,263432.41,NaN,NaN
4,apartment,|México|Veracruz de Ignacio de la Llave|Veracruz|,84.0,68508.67,19.511938,-96.871956
...,...,...,...,...,...,...
695,house,|México|Jalisco|Tlajomulco de Zúñiga|Tlajomulc...,175.0,121178.91,20.532264,-103.484418
696,house,|México|Morelos|Jiutepec|,100.0,47417.83,18.928986,-99.180215
697,house,|México|Yucatán|Mérida|,81.0,39524.23,21.028404,-89.653006
698,house,|México|San Luis Potosí|San Luis Potosí|,360.0,245050.24,22.118304,-101.032194


In [23]:
# now inspecting place_with_parent_names 
(
    pd.read_csv(path_3)
    ["place_with_parent_names"]
    .head()
)

0    |México|Distrito Federal|Gustavo A. Madero|Acu...
1             |México|Estado de México|Toluca|Metepec|
2    |México|Estado de México|Toluca|Toluca de Lerd...
3          |México|Morelos|Temixco|Burgos Bugambilias|
4    |México|Veracruz de Ignacio de la Llave|Veracruz|
Name: place_with_parent_names, dtype: object

In [24]:
(
    pd.read_csv(path_3)
    ["place_with_parent_names"]
    .str.split("|", expand=True)
    .head()
)

,0,1,2,3,4,5,6
0,,México,Distrito Federal,Gustavo A. Madero,Acueducto de Guadalupe,,None
1,,México,Estado de México,Toluca,Metepec,,None
2,,México,Estado de México,Toluca,Toluca de Lerdo,Ocho Cedros,
3,,México,Morelos,Temixco,Burgos Bugambilias,,None
4,,México,Veracruz de Ignacio de la Llave,Veracruz,,None,None


In [33]:
(
    pd.read_csv(path_3)
    ["place_with_parent_names"]
    .str.split("|", expand=True)[2]
    .head()
)

0                   Distrito Federal
1                   Estado de México
2                   Estado de México
3                            Morelos
4    Veracruz de Ignacio de la Llave
Name: 2, dtype: object

In [34]:
(
    pd.read_csv(path_3)
    .assign(
    state = lambda x: 
    x["place_with_parent_names"]
    .str.split("|", expand=True)[2]
    .head() 
    )
)

,property_type,place_with_parent_names,lat-lon,area_m2,price_usd,state
0,apartment,|México|Distrito Federal|Gustavo A. Madero|Acu...,"19.52589,-99.151703",71.0,48550.59,Distrito Federal
1,house,|México|Estado de México|Toluca|Metepec|,"19.2640539,-99.5727534",233.0,168636.73,Estado de México
2,house,|México|Estado de México|Toluca|Toluca de Lerd...,"19.268629,-99.671722",300.0,86932.69,Estado de México
3,house,|México|Morelos|Temixco|Burgos Bugambilias|,NaN,275.0,263432.41,Morelos
4,apartment,|México|Veracruz de Ignacio de la Llave|Veracruz|,"19.511938,-96.871956",84.0,68508.67,Veracruz de Ignacio de la Llave
...,...,...,...,...,...,...
695,house,|México|Jalisco|Tlajomulco de Zúñiga|Tlajomulc...,"20.532264,-103.484418",175.0,121178.91,NaN
696,house,|México|Morelos|Jiutepec|,"18.9289862,-99.1802147",100.0,47417.83,NaN
697,house,|México|Yucatán|Mérida|,"21.0284038368,-89.6530058049",81.0,39524.23,NaN
698,house,|México|San Luis Potosí|San Luis Potosí|,"22.11830417,-101.0321938992",360.0,245050.24,NaN


In [35]:
df3 = (
    pd.read_csv(path_3)
        
    .dropna()
        
    .assign(
    lat = lambda x: x["lat-lon"]
    .str.split(",", expand=True)[0]
    .astype(float),

    lon = lambda x: x["lat-lon"]
    .str.split(",", expand=True)[1]
    .astype(float), 
    
    state = lambda x: x["place_with_parent_names"]
    .str.split("|", expand=True)[2]
    .head() 
    )

    .drop(columns=["place_with_parent_names", "lat-lon"])
)

print(f'df3 is type: {type(df3)}')
print(f'df3 has a shape of: {df3.shape}')
df3.head()

df3 is type: <class 'pandas.core.frame.DataFrame'>
df3 has a shape of: (582, 6)


,property_type,area_m2,price_usd,lat,lon,state
0,apartment,71.0,48550.59,19.525890,-99.151703,Distrito Federal
1,house,233.0,168636.73,19.264054,-99.572753,Estado de México
2,house,300.0,86932.69,19.268629,-99.671722,Estado de México
4,apartment,84.0,68508.67,19.511938,-96.871956,Veracruz de Ignacio de la Llave
5,house,175.0,102763.00,20.689157,-103.366728,Jalisco


### Checking df1, df2 and df3 All Have The Same Columns Before Combining Them 

In [36]:
columns_match = set(df1.columns) == set(df2.columns) == set(df3.columns)

# the H_o = They all have the same columns 

if columns_match:

    print(f'ALL DataFrames => have the same columns.')

else: 

    print(f'Column mismatch detected')

    print("df1 vs df2 diff:", set(df1.columns) ^ set(df2.columns))         # Optional: See the difference
    print("df2 vs df3 diff:", set(df2.columns) ^ set(df3.columns))

ALL DataFrames => have the same columns.


### `pd.concat()` - Stacking Same-Schema DataFrames 

In [29]:
df = pd.concat([df1, df2, df3], ignore_index = True)

print(f'Combined Shape: {df.shape}')
df.head()
df.isnull().sum()

Combined Shape: (1736, 6)


property_type      0
state            577
lat                0
lon                0
area_m2            0
price_usd          0
dtype: int64

In [30]:
df.to_csv("./data/mexico-real-estate-combined-clean.csv", index=False)

**After `pd.concat()` — what to verify**
>
> **1 — Row count:** `df.shape[0]` should equal `df1.shape[0] + df2.shape[0] +
> df3.shape[0]`. This is the most direct sanity check: if any rows were lost or
> duplicated, the sum will not match.
>
> **2 — Column count:** `df.shape[1]` should still be 6. If `pd.concat()` found
> any column mismatch, it would have added `NaN`-filled columns — check
> `df.isnull().sum()` to confirm no new missing values appeared.
>
> **3 — Index:** Run `df.index` — with `ignore_index=True`, you should see
> `RangeIndex(start=0, stop=N, step=1)` — a clean sequential index from 0 to N-1.

**Why `ignore_index=True` matters — and what happens without it**
>
> By default, `pd.concat()` preserves each DataFrame's original index.
> Since all three files start their index at 0, the combined DataFrame will have
> *triplicate index values* — three rows with index 0, three with index 1, etc.
>
**Problems caused by duplicate index values:**
> - `df.loc[0]` returns *three rows* instead of one — unexpected behavior.
> - `.join()` and `.merge()` operations use the index to align data and will
>   produce cross-products of the duplicated rows — almost certainly wrong.
> - Some iteration patterns assume unique index values and silently compute wrong
>   results.
>
> `ignore_index=True` discards all three original indices and assigns a fresh
> continuous index (`0, 1, 2, ..., N-1`) to the combined DataFrame. This is
> almost always what you want after a vertical stack.

**One caveat:** `pd.concat()` does not check for schema mismatches — it
> silently fills mismatched columns with `NaN`. If one DataFrame has an extra
> column, the combined DataFrame will have it too, with `NaN` in the rows from
> the other DataFrames. The column-equality check we ran above prevents this.

In [37]:
if df.shape[0] == df1.shape[0] + df2.shape[0] + df3.shape[0]: print(f'all rows (N) sum to to the value of: {df.shape[0]}')

all rows (N) sum to to the value of: 1736


### Descriptive Statistics 

In [38]:
df.describe()

,lat,lon,area_m2,price_usd
count,1736.000000,1736.000000,1736.000000,1736.000000
mean,20.765410,-98.798575,170.261521,117436.962442
std,2.743425,4.882553,80.594539,66673.456313
min,15.752900,-117.054763,60.000000,33192.480000
25%,19.275200,-100.392553,101.750000,66468.430000
50%,19.620518,-99.204001,156.000000,100946.980000
75%,21.073428,-98.245911,220.000000,154911.430000
max,32.665619,-86.767539,385.000000,338888.888889


In [41]:
# Summarize the numeric columns
summary = df[["price_usd", "area_m2"]].describe()

# # Format for readability
pd.options.display.float_format = '{:,.2f}'.format

summary

,price_usd,area_m2
count,"1,736.00","1,736.00"
mean,"117,436.96",170.26
std,"66,673.46",80.59
min,"33,192.48",60.00
25%,"66,468.43",101.75
50%,"100,946.98",156.00
75%,"154,911.43",220.00
max,"338,888.89",385.00
